# Phase 6 — XLS-R Fine-Tuning on Speaker-Diverse Nepali (Kaggle)

Kaggle variant of the Colab notebook -- no Google Drive dependency. Free GPU
(T4 x2 or P100, ~30hrs/week quota), and `/kaggle/working` is downloadable
directly as a zip when the session ends, so the final checkpoint never has
to live in Drive or auto-push anywhere.

## One-time setup BEFORE running this notebook

The ~1.6GB of preprocessed audio (`data/processed/*.wav`, built in Phase 3)
is gitignored -- it is not in the GitHub repo, so Kaggle can't get it via
`git clone`. Upload it once as a Kaggle Dataset from your own machine:

```bash
# From your local NSTT-Lite/ directory:
pip install kaggle   # if you don't have the CLI
# https://www.kaggle.com/settings -> API -> Create New Token -> ~/.kaggle/kaggle.json
mkdir -p /tmp/nstt-audio && cp -r data/processed /tmp/nstt-audio/
cd /tmp/nstt-audio
kaggle datasets init -p .
# edit the generated dataset-metadata.json: set a title/id, e.g. "nstt-lite-processed-audio"
kaggle datasets create -p . --dir-mode zip
```

Then in the Kaggle notebook UI: **Add Input -> Your Datasets -> (the one you just created)**.
It will mount at `/kaggle/input/<your-dataset-slug>/`.

Also in notebook Settings: **Accelerator -> GPU T4 x2** (or P100), **Internet -> On**
(needed for `pip install` and `git clone`).

## Step 1: clone the code (no audio in the repo itself)

In [ ]:
%cd /kaggle/working
!git clone -b coursework-10phase https://github.com/Rbimochan/NSTT-Lite.git
%cd /kaggle/working/NSTT-Lite

## Step 2: install dependencies

In [ ]:
!pip install -q -r requirements.txt

## Step 3: point the repo at the uploaded audio dataset

**Set `KAGGLE_DATASET_SLUG` below to whatever you named it in Step 0** (check
the exact mounted path under `/kaggle/input/` in the file browser on the left
if unsure).

In [ ]:
KAGGLE_DATASET_SLUG = 'nstt-lite-processed-audio'  # <-- change this
import os
src = f'/kaggle/input/{KAGGLE_DATASET_SLUG}/processed'
assert os.path.exists(src), f'{src} not found -- check the dataset slug / that it is attached as an input'
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/processed'):
    os.symlink(src, 'data/processed')
print('data/processed ->', os.path.realpath('data/processed'))
print(len(os.listdir('data/processed')), 'files')

## Step 4: GPU check
**SCREENSHOT this cell's output** (GPU + library versions -- rubric evidence).

In [ ]:
import torch, transformers, datasets, mlflow
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE -- enable GPU in notebook Settings!')
print('torch', torch.__version__, '| transformers', transformers.__version__,
      '| datasets', datasets.__version__, '| mlflow', mlflow.__version__)
!wc -l data/manifests/train.jsonl data/manifests/val.jsonl data/manifests/test.jsonl

## Step 5: full fine-tuning run (multi-hour)

Everything under `/kaggle/working` (checkpoints + `mlruns/`) is what Kaggle
lets you download when the session ends -- no Drive, no auto-push to GitHub.

In [ ]:
!python scripts/run_xlsr_train.py --output-dir /kaggle/working/xlsr-ft

### If the Kaggle session times out mid-run

Kaggle's free-tier sessions have a wall-clock limit per session (not the same
always-on model as Colab+Drive). Before that happens, or right after a
disconnect: download the latest `checkpoint-*` folder under
`/kaggle/working/xlsr-ft/` from the Output panel, re-attach it as a new input
dataset in a fresh session, and resume:

In [ ]:
# Example, after re-attaching the downloaded checkpoint as e.g. /kaggle/input/xlsr-ft-resume/checkpoint-500:
#!python scripts/run_xlsr_train.py --output-dir /kaggle/working/xlsr-ft --resume /kaggle/input/xlsr-ft-resume/checkpoint-500

## Step 6: zip the checkpoint + MLflow runs for download

In [ ]:
!cd /kaggle/working && zip -qr xlsr-ft-result.zip xlsr-ft NSTT-Lite/mlruns
print('Download /kaggle/working/xlsr-ft-result.zip from the Output panel on the right.')
print('That zip is your only copy at this point -- move it to your own machine or Hugging Face Hub; nothing here persists automatically once the session ends.')

## Step 7: MLflow evidence (view locally after download)

Kaggle doesn't expose arbitrary ports the way Colab's `serve_kernel_port_as_window`
does, so view the run on your own machine after downloading:
```bash
unzip xlsr-ft-result.zip -d xlsr-ft-result
mlflow ui --backend-store-uri file://$(pwd)/xlsr-ft-result/NSTT-Lite/mlruns
```
**SCREENSHOT the WER curve** once it's open locally -- same rubric evidence as the Colab path.

## Done

Report back: final `eval_wer`, `global_step`, and confirm the zip downloaded.
Phase 7 (before/after x in/out-of-domain re-evaluation) runs against this
checkpoint once it's back on your own machine.